## **Filter Data (input here)**

In [ ]:
# ================= INPUT HERE (change monthly) =================
# Set the target month as "YYYY-MM" (e.g. "2026-07").
# Leave BLANK ("") to auto-use the previous calendar month.
MONTH_OVERRIDE = ""

## **Libraries and Functions**

In [ ]:
import requests
import pandas as pd
from urllib.parse import quote
import json
from datetime import datetime
import gspread as gs
import re
import numpy as np
import os
import sys
import calendar
from dateutil.relativedelta import relativedelta

# Column order for the raw Metabase output (extra columns are appended after these).
RAW_METABASE_COLUMN_ORDER = [
    "dp_id",
    "dpms_id",
    "dp_name",
    "tracking_id",
    "granular_status",
    "lodge_in_date",
    "poh_datetime",
    "poh_driver_name",
    "poh_driver_dp_id",
    "remarks",
    "completion_datetime",
    "cancellation_datetime",
    "shipper_id",
    "shipper_name",
    "mapping_type",
    "shipper_tier",
    "weight_category",
    "commission",
    "billing_weight",
    "from_city",
    "to_city",
    "insurance_fee",
    "delivery_fee",
    "submitted_weight",
    "actual_weight",
    "insurance_value",
    "directions",
    "l3_shipper",
    "sales_channel",
    "vol",
]

# Fixed drop-off commission override for specific DP IDs (special scheme).
SPECIAL_SCHEME_DP_IDS = {
    "13779": 250
}

# =====================================================
# DRIVE FOLDER / MONTH CONFIG (shared across the notebook)
# Target month comes from MONTH_OVERRIDE (top cell); blank = previous month.
# =====================================================
if MONTH_OVERRIDE.strip():
    anchor = datetime.strptime(MONTH_OVERRIDE.strip(), "%Y-%m")
else:
    anchor = datetime.today().replace(day=1) - relativedelta(days=1)

year = anchor.year
month = anchor.month
month_str = anchor.strftime("%Y-%m")
last_day = calendar.monthrange(year, month)[1]  # correct even when anchor is day=1
MONTH_FOLDER = anchor.strftime("%b-%Y")          # e.g. "Jul-2026"

DRIVE_ROOT = "/content/drive/MyDrive/Mitra Commission"
MONTH_PATH = f"{DRIVE_ROOT}/{MONTH_FOLDER}"
INTAKE_FILE = f"{MONTH_PATH}/Intake/Mitra Commission Intake {MONTH_FOLDER}.xlsx"
RAW_METABASE_DIR = f"{MONTH_PATH}/Raw Metabase"
RESULTS_DIR = f"{MONTH_PATH}/Results"
shipper_base_path = f"{RESULTS_DIR}/Shipper ID Level"
base_path_by_owner = f"{RESULTS_DIR}/Tracking ID Level/By Owner"
base_path_all_region_poh = f"{RESULTS_DIR}/Tracking ID Level/All Region POH"
base_path_all_region_non_poh = f"{RESULTS_DIR}/Tracking ID Level/All Region Non POH"

BASE_PATH_MAP = {
    "BY OWNER": base_path_by_owner,
    "ALL REGION - POH": base_path_all_region_poh,
    "ALL REGION - NON POH": base_path_all_region_non_poh,
}

# --- Static reference files (Origin City + Rate Card live here, NOT in the monthly intake) ---
REFERENCE_DIR = f"{DRIVE_ROOT}/Reference"   # change if your reference files live elsewhere
ORIGIN_CITY_FILE = f"{REFERENCE_DIR}/Mitra Origin City & SSB File.xlsx"  # reads its 'Origin City' tab
RATE_CARD_FILE = f"{REFERENCE_DIR}/Rate Card.xlsx"                       # reads 'Sheet1' (origin x dest matrix)
SCRIPT_ID_DIVISOR_TARGET = 4221   # SSB Script ID that triggers delivery_fee / 1.01 (Registered only)

# --- Credentials (only used when pulling from the Metabase API) ---
CRED_PATH = "/content/drive/MyDrive/metabase_mitra_commission/creds/service_account.json"
URL_CREDS = "https://docs.google.com/spreadsheets/d/1RJK6GFPVrourpdF91GQ1DWuxBBn2a9_SndoyraXckZ4/edit?gid=1998312404#gid=1998312404"
SHEET_CREDS = "App Password & API Keys"

print(f"Target month: {month_str}  |  Folder: {MONTH_FOLDER}")

In [ ]:
# Pull a Metabase card as a DataFrame via the JSON API.
def tarik_metabase(url, parameters, token, desc):
    headers = {"Content-Type":"application/x-www-form-urlencoded","X-Metabase-Session":token}
    payload = "parameters=" + quote(json.dumps(parameters))
    print("Pulling", desc)
    r = requests.post(url, headers=headers, data=payload)
    if r.status_code != 200:
        print(f"[{desc}] FAILED:", r.status_code, r.text[:300])
        return pd.DataFrame()
    data = r.json()
    return pd.DataFrame(data) if data else pd.DataFrame()

# Put preferred columns first, keep any extras after them.
def reorder_columns(df, preferred_order):
    existing_cols = [col for col in preferred_order if col in df.columns]
    remaining_cols = [col for col in df.columns if col not in preferred_order]
    return df.loc[:, existing_cols + remaining_cols]

# =====================================================
# SHARED SAVE HELPERS (used by every Save step)
# =====================================================
# Map a Region Split value to its output folder.
def resolve_base_path(region_split):
    if region_split not in BASE_PATH_MAP:
        raise ValueError(f"Region Split not recognized: {region_split}")
    return BASE_PATH_MAP[region_split]

# Build a non-clobbering path: name.csv, then name(1).csv, name(2).csv, ...
def generate_file_path(base_path, base_name, ext=".csv"):
    os.makedirs(base_path, exist_ok=True)
    file_path = os.path.join(base_path, base_name + ext)
    counter = 1
    while os.path.exists(file_path):
        file_path = os.path.join(base_path, f"{base_name}({counter}){ext}")
        counter += 1
    return file_path

# Save a DataFrame to a fresh (versioned) CSV and print where it went.
def save_versioned(df, base_path, base_name, ext=".csv"):
    path = generate_file_path(base_path, base_name, ext)
    df.to_csv(path, index=False)
    print(f"Saved: {path}")
    return path

# =====================================================
# REGISTERED-PRICING HELPERS (from_city / to_city / delivery_fee / insurance_fee)
# =====================================================
def norm(s):
    """Normalize a string key (strip); NaN -> None."""
    if pd.isna(s):
        return None
    return str(s).strip()

def norm_id(x):
    """Normalize an id that may arrive as 1897.0 (float) -> '1897'."""
    if pd.isna(x):
        return None
    try:
        return str(int(float(x)))
    except (ValueError, TypeError):
        return str(x).strip()

def calc_insurance_fee(insured_value):
    """Registered insurance fee: max(2500, 0.25% * insured_value) when insured > 0, else 0."""
    if pd.isna(insured_value):
        return 0
    v = pd.to_numeric(str(insured_value).replace(",", ""), errors="coerce")
    if pd.isna(v) or v <= 0:
        return 0
    return max(2500, 0.0025 * v)

def build_rate_card_lookup(path, sheet_name="Sheet1"):
    """Reshape the Rate Card origin x destination matrix into {(origin_L2, dest_L2): rate}.

    Layout (0-based): origins on row index 2 (Excel row 3) from col index 8 (Excel col 9);
    destination L2 in col index 3 (Excel col 4 'City/Regency L2') from row index 4 (Excel row 5).
    Destination rows repeat per L3; since to_city only carries L2, we keep the first row per
    dest L2 (rates are constant across L3 within an L2 in the source).
    """
    raw = pd.read_excel(path, sheet_name=sheet_name, header=None)
    origin_names = [norm(o) for o in raw.iloc[2, 8:].tolist()]
    block = raw.iloc[4:, 8:].copy()
    block.insert(0, "_dest", raw.iloc[4:, 3].map(norm).values)
    block = block[block["_dest"].notna()]
    block = block.drop_duplicates(subset="_dest", keep="first")
    block.columns = ["_dest"] + origin_names          # position 0 stays _dest, rest = origin L2 names
    long = block.melt(id_vars="_dest", var_name="origin", value_name="rate")
    long = long[long["origin"].notna()]
    long["rate"] = pd.to_numeric(long["rate"], errors="coerce")
    long = long.dropna(subset=["rate"])
    lookup = {(o, d): float(rt) for d, o, rt in zip(long["_dest"], long["origin"], long["rate"])}
    print(f"Rate card lookup built: {len(lookup)} (origin, dest) pairs "
          f"from {block['_dest'].nunique()} dest L2 x {len(origin_names)} origins.")
    return lookup

## **Load Metabase Data**

In [ ]:
# Load raw Metabase data for the month.
# Priority: use a manual export dropped in the "Raw Metabase" folder if present;
# otherwise pull the whole month from the Metabase API and cache it there.
IS_COLAB = "google.colab" in sys.modules
if IS_COLAB:
    print("Running in Google Colab")
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print("Running in LOCAL environment")

os.makedirs(RAW_METABASE_DIR, exist_ok=True)

METABASE_URL = "https://metabase.ninjavan.co/api/card/121565/query/json"
METABASE_FILE_XLSX = f"{RAW_METABASE_DIR}/mitra_commission_{month_str}.xlsx"
METABASE_FILE_CSV  = f"{RAW_METABASE_DIR}/mitra_commission_{month_str}.csv"

print(f"Target month: {month_str}  |  Raw Metabase dir: {RAW_METABASE_DIR}")

if os.path.exists(METABASE_FILE_XLSX):
    final_df = pd.read_excel(METABASE_FILE_XLSX)
    print(f"Loaded Metabase data from file: {METABASE_FILE_XLSX}  ({len(final_df)} rows)")
elif os.path.exists(METABASE_FILE_CSV):
    final_df = pd.read_csv(METABASE_FILE_CSV)
    print(f"Loaded Metabase data from file: {METABASE_FILE_CSV}  ({len(final_df)} rows)")
else:
    print("No export file found - pulling the full month from the Metabase API...")
    # The Metabase session token lives in the credentials sheet; fetched only when we actually pull.
    gc = gs.service_account_from_dict(json.load(open(CRED_PATH)))
    creds_ws = gc.open_by_url(URL_CREDS).worksheet(SHEET_CREDS)
    metabase_token = creds_ws.get_values('B2', value_render_option='FORMATTED_VALUE')[0][0].strip("'").strip('"')

    start_period = f"{year}-{month:02d}-01"
    end_period   = f"{year}-{month:02d}-{last_day:02d}"
    params = [
        {
            "id": "b7f1b24a-e445-482b-8df1-cfc03219bcf5",
            "type": "date/single",
            "value": start_period,
            "target": ["variable", ["template-tag", "start"]]
        },
        {
            "id": "956e1081-274a-44d6-9784-a619810b9a36",
            "type": "date/single",
            "value": end_period,
            "target": ["variable", ["template-tag", "end"]]
        }
    ]
    final_df = tarik_metabase(METABASE_URL, params, metabase_token, f"mitra_commission_{month_str}")
    if final_df is None or final_df.empty:
        raise ValueError("Metabase returned no data for the target month.")
    final_df["period_start"] = start_period
    final_df["period_end"] = end_period
    final_df.to_csv(METABASE_FILE_CSV, index=False)
    print(f"Pulled from API and cached -> {METABASE_FILE_CSV}  ({len(final_df)} rows)")

final_df = reorder_columns(final_df, RAW_METABASE_COLUMN_ORDER)
print(f"TOTAL rows: {len(final_df)}")

## **Create this month's result folders**

In [ ]:
# Create the month's folder tree (intake, raw metabase, results).
# =========================
# 1. FOLDER STRUCTURE
# =========================
TRACKING_ID_LEVEL = os.path.join(RESULTS_DIR, "Tracking ID Level")
SHIPPER_ID_LEVEL  = os.path.join(RESULTS_DIR, "Shipper ID Level")

TRACKING_SUBFOLDERS = [
    os.path.join(TRACKING_ID_LEVEL, "By Owner"),
    os.path.join(TRACKING_ID_LEVEL, "All Region POH"),
    os.path.join(TRACKING_ID_LEVEL, "All Region Non POH"),
]

ALL_FOLDERS = [
    os.path.join(MONTH_PATH, "Intake"),
    RAW_METABASE_DIR,
    RESULTS_DIR,
    SHIPPER_ID_LEVEL,
    TRACKING_ID_LEVEL,
] + TRACKING_SUBFOLDERS

# =========================
# 2. HELPER
# =========================
def ensure_folder(path):
    if os.path.exists(path):
        print(f"[exists] {path}")
    else:
        os.makedirs(path)
        print(f"[created] {path}")

# =========================
# 3. RUN
# =========================
for path in ALL_FOLDERS:
    ensure_folder(path)

print("Mitra Commission folder structure ready.")

## **Load External Inputs**

In [ ]:
# NOTE: external inputs now come from a single consolidated workbook (INTAKE_FILE, see next cell)
# instead of 8 separate Resource files.

In [ ]:
# Load the single intake workbook.
# All intake tabs mirror their source files: headers on ROW 1 (pandas default header=0)
intake_xls = pd.ExcelFile(INTAKE_FILE)

df_fake_shipper      = pd.read_excel(intake_xls, sheet_name="1a. Fake Order-Shipper")
df_fake_trid         = pd.read_excel(intake_xls, sheet_name="1b. Fake Order-TrID")
df_lipoh             = pd.read_excel(intake_xls, sheet_name="2. LIPOH DropOff-SH")
df_reg_type          = pd.read_excel(intake_xls, sheet_name="3. Region Split")
df_lex               = pd.read_excel(intake_xls, sheet_name="4. Manifest LEX")
df_last_mile         = pd.read_excel(intake_xls, sheet_name="5. Last Mile Volume")
df_last_mile_tracker = pd.read_excel(intake_xls, sheet_name="6. Last Mile Tracker")
df_drop_off          = pd.read_excel(intake_xls, sheet_name="7. List Drop SH")
df_list_mpm          = pd.read_excel(intake_xls, sheet_name="8. List Program MPM")
df_ssb               = pd.read_excel(intake_xls, sheet_name="SSB")  # SSB export: headers in row 1

In [ ]:
# =====================================================
# Load static reference data (Origin City map + Rate Card matrix)
# =====================================================
df_origin_city = pd.read_excel(ORIGIN_CITY_FILE, sheet_name="Origin City")
rate_lookup = build_rate_card_lookup(RATE_CARD_FILE)
print(f"Origin City rows: {len(df_origin_city)}  |  SSB rows: {len(df_ssb)}")

## **Logic Commission**

In [ ]:
# Logic Tier Regular & COD Commission
tier_table = pd.DataFrame({
    "tier": ["Bronze", "Silver", "Gold", "Platinum", "Diamond"],
    "min_volume": [1, 501, 1001, 3001, 10001],
    "max_volume": [500, 1000, 3000, 10000, float("inf")],
    "commission_pct": [0.15, 0.35, 0.40, 0.45, 0.50]
})

# Calculate Commission
def calculate_commission(row):
    lipoh = row["LI-POH"]
    region = row["region_group"]
    dp_id = str(row["dp_id"]).strip()

    # Special scheme for Mitra JKT @ Pagenjahan
    if dp_id in SPECIAL_SCHEME_DP_IDS:
        return SPECIAL_SCHEME_DP_IDS[dp_id]

    if pd.isna(lipoh):
        return 0

    # Jabodetabek & Banten
    if region == "Jabodetabek & Banten":
        if lipoh >= 0.95:
            return 250
        elif lipoh >= 0.90:
            return 225
        else:
            return 200

    # Outside Jabodetabek & Banten
    else:
        if lipoh >= 0.95:
            return 300
        elif lipoh >= 0.90:
            return 275
        else:
            return 250

## **Mitra driver-name cleanup logic**

In [ ]:
SUFFIX_RULES = {
    "parentheses": True,               # remove (xxx)
    "rb_variants": True,                # remove RB, RB A, RB B
    "capital_codes": [2, 3],            # remove KK, CB, etc.
    "custom_suffixes": [                # ADD NEW FINDINGS HERE
        "DROP SH",
        "MPM",
        "SH"
    ]
}

def clean_mitra_name(s, rules=SUFFIX_RULES):
    if pd.isna(s):
        return s

    s = str(s).strip()
    s = s.replace('（', '(').replace('）', ')')

    # 1. parentheses
    if rules.get("parentheses"):
        s = re.sub(r'\s*\([^)]*\)\s*$', '', s)

    # 2. RB variants
    if rules.get("rb_variants"):
        s = re.sub(r'\s+RB(\s+[A-Z])?\s*$', '', s, flags=re.IGNORECASE)

    # 3. capital codes
    for n in rules.get("capital_codes", []):
        s = re.sub(rf'\s+[A-Z]{{{n}}}\s*$', '', s)

    # 4. custom suffix list (NON-regex user friendly)
    for suf in rules.get("custom_suffixes", []):
        pattern = rf'\s+{re.escape(suf)}\s*$'
        s = re.sub(pattern, '', s, flags=re.IGNORECASE)

    return s.strip()


## **Clean Metabase data (tag & drop fake orders)**

Fake orders are matched by both mitra (DP x Shipper) and tracking_id.

In [ ]:
# Work on a copy of the raw Metabase data
final_df=final_df.copy()

In [ ]:
# tag fake orders at shipper_id level

final_df['shipper_dp_key'] = (
    final_df['dp_id'].astype(str)+final_df['shipper_id'].astype(str)
)

df_fake_shipper['shipper_dp_key'] = (
    df_fake_shipper['DP ID x Shipper ID'].astype(str)
)

final_df = final_df.merge(
    df_fake_shipper[["shipper_dp_key", "cek"]],
    on="shipper_dp_key",
    how="left"
)

fake_key_set = set(df_fake_shipper['shipper_dp_key'])

final_df['is_fake_stage1'] = final_df['shipper_dp_key'].isin(fake_key_set)

final_df["status_fake"] = final_df["cek"]
final_df.drop(columns=["cek"], inplace=True)




In [ ]:
# tag fake orders at tracking_id level

final_df = final_df.merge(
    df_fake_trid[["TRID", "cek"]],
    left_on="tracking_id",
    right_on="TRID",
    how="left"
)

final_df["status_fake"] = final_df["status_fake"].fillna(final_df["cek"])
final_df.drop(columns=["cek", "TRID"], inplace=True)

fake_tracking_set = set(df_fake_trid['TRID'])

final_df['is_fake_stage2'] = final_df['tracking_id'].isin(fake_tracking_set)


In [ ]:
# combined fake-order flag
final_df['is_fake_order'] = (
    final_df['is_fake_stage1']
    | final_df['is_fake_stage2']
)

In [ ]:
# Export the fake / problematic orders (they are dropped next)
df_fake = final_df.loc[final_df["is_fake_order"] == True].reset_index(drop=True)
df_fake = (
    df_fake
    .drop(columns=['is_fake_stage1', 'is_fake_stage2'])
)
df_fake.to_csv(f"{RAW_METABASE_DIR}/fake orders and problematic_{month_str}.csv", index=False)

In [ ]:
# drop rows flagged as fake
final_df = final_df.loc[~final_df['is_fake_order']].reset_index(drop=True)

### **Lookup Mitra Drop-Off from external data**

In [ ]:
# select the drop-off columns we need
df_drop_off_sub = df_drop_off[[
    "DP ID","Join Date"	,"Withdraw Date",
    "Remarks"
]].rename(columns={"Remarks": "Remarks_drop_off", "DP ID":"DP ID dari mitra drop off"}).copy()

In [ ]:
# merge raw data with the Mitra drop-off list
final_df_merged = final_df.merge(
    df_drop_off_sub,
    how="left",
    left_on="dp_id",
    right_on="DP ID dari mitra drop off"
)

In [ ]:
# check drop-off eligibility by lodge-in date vs join/withdraw window

data_cols = ['lodge_in_date', 'Join Date', 'Withdraw Date']

final_df_merged[data_cols] = final_df_merged[data_cols].apply(
    lambda col: pd.to_datetime(col, errors='coerce')
)

final_df_merged['check_drop_off_by_period'] = (
    (final_df_merged['lodge_in_date'] >= final_df_merged['Join Date']) &
    (
        final_df_merged['Withdraw Date'].isna() |
        (final_df_merged['lodge_in_date'] <= final_df_merged['Withdraw Date'])
    )
)

In [ ]:
# Drop helper columns no longer needed
final_df_merged = (
    final_df_merged
    .drop(columns=['is_fake_stage1', 'is_fake_stage2', 'DP ID dari mitra drop off'])
)

### **Calculate Drop Off Commission**

In [ ]:
# select the Li-POH columns we need
df_lipoh_sub = df_lipoh[[
    "DP ID",
    "Clean Up Region",
    "LI-POH"
]].copy()

In [ ]:
# merge raw data with Li-POH data
final_df_merged = final_df_merged.merge(
    df_lipoh_sub,
    how="left",
    left_on="dp_id",
    right_on="DP ID"
)

In [ ]:
# build region group: Jabodetabek vs non-Jabodetabek
final_df_merged["region_group"] = np.where(
    final_df_merged["Clean Up Region"]
        .str.strip()
        .str.lower()
        .eq("jabodetabek"),
    "Jabodetabek & Banten",
    "Non Jabodetabek & Banten"
)

# normalize Li-POH to numeric
final_df_merged["LI-POH"] = pd.to_numeric(
    final_df_merged["LI-POH"],
    errors="coerce"
)

In [ ]:
# calculate drop-off commission
final_df_merged["drop_off_commission"] = (
    final_df_merged.apply(calculate_commission, axis=1)
)


In [ ]:
final_df_merged['drop_off_commission'] = np.where(
    final_df_merged['check_drop_off_by_period'],
    final_df_merged['drop_off_commission'],
    0
)


#### **Rename Mitra (one-off)**

In [ ]:
# One-off rename of a mitra that changed name
final_df_merged["dp_name"] = final_df_merged["dp_name"].replace(
    "Mitra TSK @ Tugujaya",
    "Mitra TSK @ Pakemitankidul"
)


### **Lookup Region Type**

In [ ]:
# Prepare Region Split lookup (rename key to avoid a merge clash)
df_reg_type_used = (
    df_reg_type[["DP ID", "Region Split"]]
    .rename(columns={"DP ID": "DP ID 2"})
    .copy()
)

In [ ]:
# Merge Region Split; default missing to 'All Region - Non POH'
final_df_merged = final_df_merged.merge(
    df_reg_type_used,
    how='left',
    left_on='dp_id',
    right_on='DP ID 2'
)

final_df_merged['Region Split'] = final_df_merged['Region Split'].fillna('All Region - Non POH')

final_df_merged.drop(columns=['DP ID 2'], inplace=True)

In [ ]:
# Prefer the drop-off remark, fall back to the Metabase remark
final_df_merged['remarks_final'] = np.where(
    final_df_merged['Remarks_drop_off'].notna() & (final_df_merged['Remarks_drop_off'] != ''),
    final_df_merged['Remarks_drop_off'],
    final_df_merged['remarks']
)

## **Acquisition, Allocation & Unregistered Parcels**

### **Shipper-Level Acquisition, Allocation & Unregistered Parcels**

In [ ]:
# Data Population
df_non_registered = final_df_merged[
    (~final_df_merged["mapping_type"]
        .str.lower()
        .isin(["registered", "unregistered not eligible","cancelled at mitra"]))
    # (optional: also require remarks == "mitra drop off to sh")
].copy()

In [ ]:
# Aggregation/ Pivot
agg_akus_alo_unreg = (
    df_non_registered
    .groupby(
        [
            "shipper_id",
            "shipper_name",
            "dp_id",
            "dp_name",
            "mapping_type"
        ],
        as_index=False
    )
    .agg(
        total_tracking_id=("tracking_id", "count"),
        total_nv_weight=("billing_weight", "sum"),
        total_commission=("commission", "sum"),
        total_parcel_drop_off=("check_drop_off_by_period", "sum"),  # count of parcels actually dropped off
        total_drop_off_commission=(
            "drop_off_commission",
            lambda x: x[df_non_registered.loc[x.index, "check_drop_off_by_period"]].sum()
        )  # commission only for check_drop_off_by_period == True
    )
)

In [ ]:
save_versioned(agg_akus_alo_unreg, shipper_base_path, "Shipper Level Akuisisi, Alokasi, dan Unregistered Parcel")

### **Tracking-ID-Level Acquisition, Allocation & Unregistered Parcels**

In [ ]:
# Data Population
df_non_registered_trid = (
    final_df_merged[['dp_id', 'dp_name','shipper_id','shipper_name','tracking_id','billing_weight','weight_category','from_city','to_city','lodge_in_date','completion_datetime','granular_status','shipper_tier','commission','Remarks_drop_off','check_drop_off_by_period','drop_off_commission','mapping_type','Region Split']]
    .rename(columns={'billing_weight': "weight"})
)
for col in ["mapping_type", "Region Split"]:
    df_non_registered_trid[col] = (
        df_non_registered_trid[col]
        .astype(str)
        .str.strip()
        .str.upper()
    )

In [ ]:
# filter raw data

valid_region_split = ["BY OWNER", "ALL REGION - POH","ALL REGION - NON POH"]

df_non_registered_trid_clean = df_non_registered_trid[
    (df_non_registered_trid["Region Split"].isin(valid_region_split)) &
    (df_non_registered_trid["mapping_type"] != "REGISTERED")
    # (optional: also require Remarks_drop_off == "mitra drop off to sh")
].copy()

In [ ]:
# Save by Region Split
BASE_NAME_MAP = {
    "ACQUISITION": "Tracking id Level Akuisisi Parcel",
    "ALLOCATION": "Tracking id Level Alokasi Parcel",
    "UNREGISTERED ELIGIBLE": "Tracking id Level Unregistered Eligible Parcel",
    "UNREGISTERED NOT ELIGIBLE": "Tracking id Level Unregistered Not Eligible Parcel",
    "CANCELLED AT MITRA": "Tracking id Level Cancelled At Mitra Parcel",
}

for (mapping_type, region_split), df_sub in (
    df_non_registered_trid_clean
    .groupby(["mapping_type", "Region Split"])
):
    if mapping_type not in BASE_NAME_MAP:
        raise ValueError(f"mapping_type not recognized: {mapping_type}")
    save_versioned(df_sub, resolve_base_path(region_split), BASE_NAME_MAP[mapping_type])

## **COD & Regular Parcels**

### **Shipper Level COD Regular Commission**

In [ ]:
# Data Population

df_registered = final_df_merged[
    (final_df_merged["mapping_type"]
        .str.lower()
        .isin(["registered"]))
    # (optional: also require remarks == "mitra drop off to sh")
].copy()

In [ ]:
df_registered["remarks_cod_regular"] = (
    df_registered["shipper_name"]
    .astype(str)
    .str.contains(r"\bCOD\b", case=False, regex=True)
    .map({True: "COD", False: "Regular"})
)

In [ ]:
df_registered['delivery_fee'] = (
    df_registered['delivery_fee']
    .replace({',': ''}, regex=True)
    .pipe(pd.to_numeric, errors='coerce')
)

In [ ]:
# =====================================================
# REGISTERED-ONLY overrides: from_city, to_city, delivery_fee, insurance_fee
# (applied after delivery_fee is cleaned, before aggregation)
# =====================================================
# Lookup maps from reference/SSB data
origin_map     = {norm_id(k): norm(v) for k, v in zip(df_origin_city["DP ID"], df_origin_city["Origin Update"])}
ssb_l2_map     = {norm(k): norm(v)    for k, v in zip(df_ssb["Tracking ID"], df_ssb["L2 Name"])}
ssb_script_map = {norm(k): norm_id(v) for k, v in zip(df_ssb["Tracking ID"], df_ssb["Script ID"])}

_dp_key   = df_registered["dp_id"].map(norm_id)
_trid_key = df_registered["tracking_id"].map(norm)

# 1) from_city <- Origin City "Origin Update" (keep original on miss)
_new_from = _dp_key.map(origin_map)
df_registered["from_city"] = _new_from.where(_new_from.notna(), df_registered["from_city"])

# 2) to_city <- SSB "L2 Name" (keep original on miss)
_new_to = _trid_key.map(ssb_l2_map)
df_registered["to_city"] = _new_to.where(_new_to.notna(), df_registered["to_city"])

# 3) delivery_fee - DECISION (2026-07): use the RAW Metabase fee.
#    The June reconciliation proved the Rate Card override cut Registered/Regular fees 26-80%
#    below raw, while raw == the manual June file exactly (e.g. DP 18937 = 772,800). So use raw;
#    keep the Rate Card only as a FALLBACK for blank/zero raw fees. Flip USE_RATE_CARD_FEE=True
#    only after the rate card is validated.
USE_RATE_CARD_FEE = False
_orig_fee = pd.to_numeric(df_registered["delivery_fee"], errors="coerce")
_pairs = list(zip(df_registered["from_city"].map(norm), df_registered["to_city"].map(norm)))
_rc_fee = pd.Series([rate_lookup.get(p) for p in _pairs], index=df_registered.index)
df_registered["_rate_card_hit"] = _rc_fee.notna()
if USE_RATE_CARD_FEE:
    df_registered["delivery_fee"] = _rc_fee.where(_rc_fee.notna(), _orig_fee)   # rate card primary
else:
    df_registered["delivery_fee"] = _orig_fee.where(_orig_fee.notna() & (_orig_fee > 0), _rc_fee)  # raw primary, rate card fallback

# 4) Script ID adjustment: if SSB Script ID == target -> delivery_fee / 1.01
_script = _trid_key.map(ssb_script_map)
_is_target = _script.eq(str(SCRIPT_ID_DIVISOR_TARGET))
df_registered.loc[_is_target, "delivery_fee"] = df_registered.loc[_is_target, "delivery_fee"] / 1.01

# 5) insurance_fee <- max(2500, 0.25% * insurance_value) when insured > 0 else 0
df_registered["insurance_fee"] = df_registered["insurance_value"].apply(calc_insurance_fee)

n = len(df_registered)
print("Registered overrides applied:")
print(f"  from_city filled from Origin City : {int(_new_from.notna().sum())} / {n}")
print(f"  to_city   filled from SSB L2 Name : {int(_new_to.notna().sum())} / {n}")
print(f"  delivery_fee source               : {'RATE CARD' if USE_RATE_CARD_FEE else 'RAW Metabase'} "
      f"(rate-card would hit {int(df_registered['_rate_card_hit'].sum())} / {n}; used only as fallback for blank raw)")
print(f"  script_id == {SCRIPT_ID_DIVISOR_TARGET} (/1.01 applied) : {int(_is_target.sum())}")

In [ ]:
# =====================================================
# VALIDATION: no blank delivery_fee for Registered parcels
# =====================================================
_miss_ct = int((~df_registered["_rate_card_hit"]).sum())
print(f"Rate card misses (kept original delivery_fee): {_miss_ct} / {len(df_registered)}")
if _miss_ct:
    _unmatched = (
        df_registered.loc[~df_registered["_rate_card_hit"], ["from_city", "to_city"]]
        .drop_duplicates()
    )
    print(f"  distinct unmatched (origin, dest) city pairs: {len(_unmatched)} (showing up to 30)")
    print(_unmatched.head(30).to_string(index=False))

_blank = df_registered[df_registered["delivery_fee"].isna()]
if len(_blank):
    print(f"\n[STOP] BLANK delivery_fee for {len(_blank)} Registered parcels. Sample tracking_ids:")
    print(_blank["tracking_id"].head(20).tolist())
    raise ValueError(
        "Registered parcels have blank delivery_fee - fix source data (Rate Card / Metabase) before continuing."
    )
else:
    print("\nOK: no blank delivery_fee for Registered parcels.")

# drop helper column so it does not leak downstream
df_registered = df_registered.drop(columns=["_rate_card_hit"])

In [ ]:
# Aggregation/ Pivot
agg_cod = (
    df_registered
    .groupby(
        ["shipper_id", "shipper_name", "dp_id", "dp_name", "mapping_type", "remarks_cod_regular"],
        as_index=False
    )
    .agg(
        total_tracking_id=("tracking_id", "count"),
        total_nv_weight=("billing_weight", "sum"),
        total_delivery_fee=("delivery_fee", "sum"),
        total_parcel_drop_off=("check_drop_off_by_period", "sum"),
        total_insurance_fee=("insurance_fee", "sum"),
        total_drop_off_commission=(
            "drop_off_commission",
            lambda x: x[df_registered.loc[x.index, "check_drop_off_by_period"]].sum()
        )  # commission only for check_drop_off_by_period == True
    )
)


In [ ]:
# Get Tiering for Commission
def get_tier(volume):
    row = tier_table[
        (tier_table["min_volume"] <= volume) &
        (volume <= tier_table["max_volume"])
    ]
    if row.empty:
        return pd.Series([None, 0])
    return pd.Series([row.iloc[0]["tier"], row.iloc[0]["commission_pct"]])

In [ ]:
# Calculate Commission
agg_cod[["tier", "commission_pct"]] = agg_cod["total_tracking_id"].apply(get_tier)
agg_cod["commission"] = (
    agg_cod["total_delivery_fee"] * agg_cod["commission_pct"]
)


In [ ]:
save_versioned(agg_cod, shipper_base_path, "Shipper Level Regular & COD Parcel")

### **Tracking id Level COD Commission**

In [ ]:
# Data Population
df_registered_trid = (
    df_registered[['dp_id', 'dp_name','shipper_id','shipper_name','tracking_id','billing_weight','from_city','to_city','lodge_in_date','completion_datetime','granular_status','delivery_fee', 'insurance_fee','commission','remarks','drop_off_commission','mapping_type','Region Split','remarks_cod_regular']]
    .rename(columns={'billing_weight': "weight"})
)

In [ ]:
# Tier per dp_id from its total tracking-id count
agg_tier = (
    df_registered_trid
    .groupby(
        ["dp_id"],
        as_index=False
    )
    .agg(
        total_tracking_id=("tracking_id", "count")))
agg_tier[["tier", "commission_pct"]] = agg_tier["total_tracking_id"].apply(get_tier)

In [ ]:
# Attach the per-dp tier to each tracking-id row
df_registered_trid = df_registered_trid.merge(
    agg_tier,
    how="left",
    left_on="dp_id",
    right_on="dp_id"
)

In [ ]:
# Per-parcel commission = delivery_fee x tier %
df_registered_trid["commission"] = (
    df_registered_trid["delivery_fee"] * df_registered_trid["commission_pct"]
)

In [ ]:
df_registered_trid = (
    df_registered_trid
    .drop(columns=['total_tracking_id'])
)

In [ ]:
for col in ["Region Split"]:
    df_registered_trid[col] = (
        df_registered_trid[col]
        .astype(str)
        .str.strip()
        .str.upper()
    )

In [ ]:
VALID_REGION = [
    "ALL REGION - POH",
    "ALL REGION - NON POH",
    "BY OWNER"
]

df_registered_trid = df_registered_trid[
    df_registered_trid["Region Split"].isin(VALID_REGION)
].copy()


In [ ]:
# Save by Region Split
BASE_NAME_MAP = {
    "COD": "Tracking id Mitra COD Parcel",
    "Regular": "Tracking id Mitra Regular Parcel",
}

for (remarks_cod_regular, region_split), df_sub in (
    df_registered_trid
    .groupby(["remarks_cod_regular", "Region Split"])
):
    if remarks_cod_regular not in BASE_NAME_MAP:
        raise ValueError(f"remarks_cod_regular not recognized: {remarks_cod_regular}")
    save_versioned(df_sub, resolve_base_path(region_split), BASE_NAME_MAP[remarks_cod_regular])

## **Lex Parcel**

### **Shipper Level Lex Hub Pickup Parcel**



In [ ]:
# Aggregation/ Pivot
agg_lex = (
    df_lex
    .groupby(
        ["global_shipper_id", "shipper_name", "DPID", "Mitra Name"],
        as_index=False
    )
    .agg(
        total_tracking_id=("List Tracking Id", "count"),
        total_commission=("Commission", "sum")
    )
)


In [ ]:
save_versioned(agg_lex, shipper_base_path, "Shipper Level Lex Hub Pickup Parcel")

### **Tracking id Lex Parcel**

In [ ]:
# Data Population

df_lex_trid = df_lex.merge(
    df_reg_type_used,
    how='left',
    left_on='DPID',
    right_on='DP ID 2'
)

In [ ]:
df_lex_trid['Region Split'] = df_lex_trid['Region Split'].fillna('All Region - Non POH')

df_lex_trid.drop(columns=['DP ID 2'], inplace=True)

In [ ]:
for col in ["Region Split"]:
    df_lex_trid[col] = (
        df_lex_trid[col]
        .astype(str)
        .str.strip()
        .str.upper()
    )

In [ ]:
BASE_FILE_NAME = "Tracking id LEX Hub Pickup Parcel"

for region_split, df_sub in df_lex_trid.groupby("Region Split"):
    save_versioned(df_sub, resolve_base_path(region_split), BASE_FILE_NAME)

## **MPM Parcel**

### **Lookup Mitra-Pickup-Mitra from external data**

In [ ]:
# keep only the MPM columns we need
df_mpm_sub = (
    df_list_mpm
    .drop(columns=['Status', 'Status.1', 'Start Date', 'Offboard Date','Note Drop Out'])
    .rename(columns={'Remarks': 'Remarks_mpm'})
)

df_mpm_sub['dpid_for__mitra_by'] = (
    df_mpm_sub['DPID (Pickup for)'].astype(str).str.strip()
    + df_mpm_sub['Mitra Name (Pickup by)'].astype(str).str.strip()
)


### **Shipper Level MPM Parcel**

In [ ]:
# Start the MPM flow from a fresh copy of the clean data
df_mpm = final_df.copy()

In [ ]:
df_mpm['poh_driver_name'] = df_mpm['poh_driver_name'].apply(clean_mitra_name)

In [ ]:
# Build MPM merge key: dp_id + cleaned POH driver name
df_mpm['helper']= (
    df_mpm['dp_id'].astype(str)+df_mpm['poh_driver_name'].astype(str)
)

In [ ]:
# Merge with the MPM program list on that key
df_mpm_merged = df_mpm.merge(
    df_mpm_sub,
    how="left",
    left_on="helper",
    right_on="dpid_for__mitra_by"
)


In [ ]:
df_mpm_merged  = df_mpm_merged [
    (df_mpm_merged ["Remarks_mpm"]
        .str.lower()
        .isin(["mitra pickup mitra"]))]

In [ ]:
df_lipoh_mpm = df_lipoh[[
    "Mitra Name",
    "DP ID",
    "Clean Up Region",
    "LI-POH"
]].copy()

In [ ]:
# Bring in Li-POH for the pickup mitra
df_mpm_merged_final = df_mpm_merged.merge(
    df_lipoh_mpm,
    how="left",
    left_on="poh_driver_name",
    right_on="Mitra Name"
)


In [ ]:
# build region group: Jabodetabek vs non-Jabodetabek
df_mpm_merged_final["region_group"] = np.where(
    df_mpm_merged_final["Clean Up Region"]
        .str.strip()
        .str.lower()
        .eq("jabodetabek"),
    "Jabodetabek & Banten",
    "Non Jabodetabek & Banten"
)

# normalize Li-POH to numeric
df_mpm_merged_final["LI-POH"] = pd.to_numeric(
    df_mpm_merged_final["LI-POH"],
    errors="coerce"
)

In [ ]:
# calculate drop-off commission
df_mpm_merged_final["drop_off_commission"] = (
    df_mpm_merged_final.apply(calculate_commission, axis=1)
)

In [ ]:
# Aggregation/ Pivot
agg_mpm = (
    df_mpm_merged_final
    .groupby(
        ["shipper_id", "shipper_name", 'DPID (Pickup by)', 'Mitra Name (Pickup by)','Remarks_mpm'],
        as_index=False
    )
    .agg(
        total_tracking_id=("tracking_id", "count"),
        total_commission=("drop_off_commission","sum")
    )
)


In [ ]:
save_versioned(agg_mpm, shipper_base_path, "Shipper Level Mitra Pickup Mitra")

### **Tracking id Level Mitra Pickup Mitra (MPM)**

In [ ]:
# Data Population

df_mpm_trid = df_mpm_merged_final.merge(
    df_reg_type_used,
    how='left',
    left_on='DPID (Pickup by)',
    right_on='DP ID 2'
)

In [ ]:
df_mpm_trid['Region Split'] = df_mpm_trid['Region Split'].fillna('All Region - Non POH')

df_mpm_trid.drop(columns=['DP ID 2'], inplace=True)

In [ ]:
for col in ["Region Split"]:
    df_mpm_trid[col] = (
        df_mpm_trid[col]
        .astype(str)
        .str.strip()
        .str.upper()
    )

In [ ]:
# Data Population
df_mpm_trid = (
    df_mpm_trid[['dp_id', 'dp_name','DPID (Pickup by)', 'Mitra Name (Pickup by)','tracking_id','shipper_id','shipper_name','Remarks_mpm','drop_off_commission','Region Split']]
    .rename(columns={'DP ID': "origin_id",
                     'Mitra Name':"origin_name"})
)

In [ ]:
BASE_FILE_NAME = "Tracking id Mitra Pickup Mitra Parcel"

for region_split, df_sub in df_mpm_trid.groupby("Region Split"):
    save_versioned(df_sub, resolve_base_path(region_split), BASE_FILE_NAME)

## **Last Mile Parcel**

### **Shipper Level Last Mile Parcel**

In [ ]:
# Keep only the Last Mile tracker columns we need
df_Last_mile_tracker_need = df_last_mile_tracker[[
    "hub_name",
    "Region",
    "Final Tier",
    'PPP Reg',
    'PPP RTS Reg',
    'PPP Bulky',
    'PPP RTS Bulky',
    'PPP Superbulky',
    'PPP RTS Superbulky'
]].copy()

In [ ]:
# Join last-mile volume with per-hub PPP rates
df_last_mile_final = df_last_mile.merge(
    df_Last_mile_tracker_need,
    on='hub_name',
    how='left'
)


In [ ]:
def get_commission(row):
    status = row['granular_status'].lower()
    size = row['size_category'].lower()

    # only count eligible statuses
    if status not in ['completed', 'returned to sender']:
        return 0

    if size == 'regular':
        return row['PPP Reg'] if status == 'completed' else row['PPP RTS Reg']

    elif size == 'bulky':
        return row['PPP Bulky'] if status == 'completed' else row['PPP RTS Bulky']

    elif size == 'superbulky':
        return row['PPP Superbulky'] if status == 'completed' else row['PPP RTS Superbulky']

    else:
        return 0


In [ ]:
# Commission per parcel (status x size)
df_last_mile_final['commission'] = df_last_mile_final.apply(get_commission, axis=1)


In [ ]:
# Aggregation/ Pivot
agg_lm = (
    df_last_mile_final
    .groupby(
        ["shipper_id", "shipper_name", "dp_id", "dp_name"],
        as_index=False
    )
    .agg(
        total_tracking_id=("tracking_id", "count"),
        total_commission=("commission","sum")
    )
)


In [ ]:
save_versioned(agg_lm, shipper_base_path, "Shipper Level Last Mile Parcel")

### **Tracking id Level Last Mile Parcel**

In [ ]:
# Data Population

df_last_mile_final_trid = df_last_mile_final.merge(
    df_reg_type_used,
    how='left',
    left_on='dp_id',
    right_on='DP ID 2'
)

In [ ]:
df_last_mile_final_trid['Region Split'] = df_last_mile_final_trid['Region Split'].fillna('All Region - Non POH')

df_last_mile_final_trid.drop(columns=['DP ID 2'], inplace=True)

In [ ]:
for col in ["Region Split"]:
    df_last_mile_final_trid[col] = (
        df_last_mile_final_trid[col]
        .astype(str)
        .str.strip()
        .str.upper()
    )

In [ ]:
# Data Population
df_last_mile_final_trid = (
    df_last_mile_final_trid[['dp_id', 'dp_name','tracking_id','shipper_id','shipper_name','completion_date','granular_status','size_category','Final Tier','commission','Region Split']]
)

In [ ]:
BASE_FILE_NAME = "Tracking id Last Mile Parcel"

for region_split, df_sub in df_last_mile_final_trid.groupby("Region Split"):
    save_versioned(df_sub, resolve_base_path(region_split), BASE_FILE_NAME)

## **Pivot All Region (Phase 1: Data → Pivot Data → Commission)**

Builds `M##Y#### Pivot All Region.xlsx` from this run's shipper-level aggregates plus the all-region KPI intake, stamping computed **values** into a copy of the prior-month workbook (embedded pivots/formulas become static values).

**Prereqs:** upload `par_pipeline.py` and `par_writer.py` to `{DRIVE_ROOT}/automation`, provide the all-region KPI intake, and have the prior-month `Pivot All Region.xlsx` available as the template. Verified engine parity: `automation/verify_commission_engine.py` (0 mismatches on all computed columns).

In [ ]:
# === Pivot All Region (Phase 1 + 2a) - config & imports ===================
import sys, os, glob

# par_pipeline.py + par_writer.py + par_adjustment.py + par_format_sr.py on Drive.
AUTOMATION_DIR = f"{DRIVE_ROOT}/automation"
if AUTOMATION_DIR not in sys.path:
    sys.path.insert(0, AUTOMATION_DIR)
import importlib
import par_pipeline as par
import par_writer
import par_adjustment as par_adj
import par_format_sr as par_fsr
for _m in (par, par_writer, par_adj, par_fsr):
    importlib.reload(_m)

# This month's output workbook: M{mm}Y{yyyy} Pivot All Region.xlsx
PAR_OUTPUT_FILE = f"{MONTH_PATH}/M{anchor.strftime('%m')}Y{anchor.strftime('%Y')} Pivot All Region.xlsx"

# SINGLE master intake (KPI + manual-row + Adjustment tabs), exported from the master Gsheet.
ALL_REGION_INTAKE_FILE = INTAKE_FILE

def _find1(*globs):
    for g in globs:
        hits = sorted(glob.glob(g))
        if hits:
            return hits[0]
    return None

# Phase 2a penalty inputs. PRIMARY: the standardized "Penalty Intake" workbook (4 near-raw tabs,
# exported from the shared Penalty Intake Gsheet). FALLBACK: the 3 raw stakeholder exports dropped
# in a "Penalty SR" folder. The Outstanding block (L:Y) comes from the Penalty Intake's Outstanding
# tab, else a compiled Format SR / finance file.
PENALTY_INTAKE_FILE = _find1(f"{MONTH_PATH}/Intake/Penalty Intake*.xlsx",
                             f"{MONTH_PATH}/Penalty Intake*.xlsx",
                             f"{DRIVE_ROOT}/Penalty Intake*.xlsx")

PENALTY_SR_DIR = f"{MONTH_PATH}/Penalty SR"
PPDP_FILE    = _find1(f"{PENALTY_SR_DIR}/Penalty PPDP*.xlsx", f"{DRIVE_ROOT}/Penalty SR/Penalty PPDP*.xlsx")
PODPODA_FILE = _find1(f"{PENALTY_SR_DIR}/*Fake POD*PODA*.xlsx", f"{PENALTY_SR_DIR}/*Fake POD*.xlsx",
                      f"{DRIVE_ROOT}/Penalty SR/*Fake POD*.xlsx")
MISSING_FILE = _find1(f"{PENALTY_SR_DIR}/Penalty LM*.xlsx", f"{DRIVE_ROOT}/Penalty SR/Penalty LM*.xlsx")

# Outstanding (L:Y) fallback source if the Penalty Intake workbook is absent.
FORMAT_SR_FILE = _find1(f"{MONTH_PATH}/Format SR Penalty*.xlsx", f"{DRIVE_ROOT}/Format SR Penalty*.xlsx")

# Inactive-hub -> DP overrides (hubs with no LM-penalty history, so absent from the POD/PODA
# hub->DP map). Maintain as new inactive hubs appear; the compiler surfaces any unmapped rows.
HUB_DP_OVERRIDES = {
    "DP-KOI-GDCEK": (50231, "Mitra JKT @ Cengkareng"),
    "DP-MAC-GDTLN": (2621, "Mitra JKT @ Gondrong"),
}

# Prior-month workbook is the writer TEMPLATE (values stamped into a copy of it).
_prev = (anchor.replace(day=1) - pd.Timedelta(days=1))
PREV_FOLDER = _prev.strftime("%b-%Y")
PREV_PAR_FILE = f"{DRIVE_ROOT}/{PREV_FOLDER}/M{_prev.strftime('%m')}Y{_prev.strftime('%Y')} Pivot All Region.xlsx"

print("Master intake  :", ALL_REGION_INTAKE_FILE)
print("Penalty Intake :", PENALTY_INTAKE_FILE, "  <- primary penalty source")
print("Penalty SR (raw fallback):", PPDP_FILE, "|", PODPODA_FILE, "|", MISSING_FILE)
print("Format SR (OS fallback)  :", FORMAT_SR_FILE)
print("Template (prev):", PREV_PAR_FILE, "| exists:", os.path.exists(PREV_PAR_FILE))
print("Output         :", PAR_OUTPUT_FILE)

In [ ]:
# === Pivot All Region (Phase 1 + 2a) - build & write ======================
# Shipper-level inputs: reuse the in-memory aggregates produced above (no re-read).
shipper = {
    "akus_alo_unreg": agg_akus_alo_unreg,
    "regular_cod":    agg_cod,
    "last_mile":      agg_lm,
    "lex":            agg_lex,
    "mpm":            agg_mpm,
}

# All-region KPI + manual-row + adjustment intake tabs.
ar_xls   = pd.ExcelFile(ALL_REGION_INTAKE_FILE)
df_acq   = pd.read_excel(ar_xls, sheet_name="Acquisition KPI")
df_allo  = pd.read_excel(ar_xls, sheet_name="Allocation KPI")
df_unreg = pd.read_excel(ar_xls, sheet_name="Unreg KPI")

def _opt_tab(name):
    try:
        return pd.read_excel(ar_xls, sheet_name=name)
    except Exception:
        return None

df_cross_border = _opt_tab("Cross Border")   # manual XB rows (30% x delivery fee)
df_claims       = _opt_tab("Claims")         # prior-month dispute/claim rows
df_adjustment   = _opt_tab("Adjustment")     # approved commission adjustments (E/F/G/J)

# Claim KPI (G,O,AJ,AX) come from the previous month's pivot (value - 100%); mostly 0.
prev_claim_kpi = None

# --- Phase 1 sheets -----------------------------------------------------------
data = par.build_data_sheet(shipper, df_reg_type,
                            df_cross_border=df_cross_border, df_disputes=df_claims)
kpi_fn = par.build_kpi_row_fn(
    df_acq=df_acq, df_allo=df_allo, df_unreg=df_unreg,
    df_lipoh=df_lipoh, df_lm_tracker=df_last_mile_tracker,
    prev_claim_kpi=prev_claim_kpi,
)
commission = par.assemble_commission(data, kpi_fn)
print(f"Data rows: {len(data)} | Commission rows (DPIDs): {len(commission)}")

# --- Phase 2b: Data Sources feed + Rekap / By-Owner key lists -----------------
df_data_changes = _opt_tab("Data Mitra Changes")   # mitra-master upsert feed (by DPID)
rekap_keys   = par.build_rekap_keys(commission)     # unique-mitra keys (Rekap col A)
byowner_keys = par.build_byowner_keys(commission)   # By-Owner DPIDs (By Owner col A)
print(f"Rekap keys: {len(rekap_keys)} | By-Owner keys: {len(byowner_keys)}")

# --- Phase 2a: compile the penalty detail (H/I) + outstanding (L:Y) -----------
# Prefer the standardized Penalty Intake workbook; fall back to the raw Penalty SR files.
penalty_detail, outstanding_df = None, None
if PENALTY_INTAKE_FILE:
    penalty_detail = par_fsr.build_penalty_detail_from_intake(
        PENALTY_INTAKE_FILE, hub_dp_overrides=HUB_DP_OVERRIDES)
    outstanding_df = par_fsr.read_intake_outstanding(PENALTY_INTAKE_FILE)
elif PPDP_FILE and PODPODA_FILE and MISSING_FILE:
    penalty_detail = par_fsr.build_penalty_detail(
        PPDP_FILE, PODPODA_FILE, MISSING_FILE, hub_dp_overrides=HUB_DP_OVERRIDES)
elif FORMAT_SR_FILE:
    print("No Penalty Intake / raw files -> using compiled Format SR file for H/I.")

if penalty_detail is not None:
    _um = penalty_detail.attrs.get("unmapped_missing")
    if _um is not None and len(_um):
        print(f"ACTION: {len(_um)} Missing rows unmapped to a DP -> add their hub to "
              f"HUB_DP_OVERRIDES and re-run:")
        display(_um)

# Adjustment & Penalty sheet (needs the built Data for the DPMS/Region VLOOKUP).
adj_result = None
if penalty_detail is not None or FORMAT_SR_FILE or df_adjustment is not None:
    adj_result = par_adj.build_adjustment_penalty(
        data, df_adjustment,
        format_sr_path=(None if penalty_detail is not None else FORMAT_SR_FILE),
        penalty_detail=penalty_detail,
        outstanding_df=outstanding_df,
        outstanding_path=(None if outstanding_df is not None else FORMAT_SR_FILE))
else:
    print("WARN: no penalty inputs -> Adjustment&Penalty sheet left as template.")

# --- Write a LIVE copy: pivots + formulas preserved, refreshes on open (~3 min) --
assert os.path.exists(PREV_PAR_FILE), (
    f"Template not found: {PREV_PAR_FILE}. Upload the prior-month 'Pivot All Region.xlsx' to copy from.")
par_writer.write_pivot_all_region(PREV_PAR_FILE, PAR_OUTPUT_FILE, data, commission,
                                  adj_result=adj_result,
                                  data_sources_changes=df_data_changes,
                                  rekap_keys=rekap_keys, byowner_keys=byowner_keys,
                                  month=anchor.strftime("%Y-%m"))
print("Wrote:", PAR_OUTPUT_FILE)